In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric.nn as geom_nn
import os
import logging
from tqdm import tqdm
import numpy as np
from sklearn.metrics import precision_score,accuracy_score,recall_score,f1_score,roc_auc_score
from wsi.nets.vision_transformer import vit_small,Block
from wsi.wsi import *
from wsi.utils import *
from wsi.nets.dasmil import *

from torch_geometric.data import Data
from sklearn.model_selection import train_test_split
from nystrom_attention import NystromAttention


In [ ]:
def get_data(path,device):
    data = torch.load(path)
    features = data["features"].to(device)
    label = torch.tensor([data["label"]]).to(device)
    rects = {(x,y):i for i,(x,y) in enumerate(data["rects"])}
    edges = []
    s = data["patch_size"]
    for i,(x,y) in enumerate(rects):
        for x_i,y_i in [(x+s,y-s),(x+s,y),(x+s,y+s),(x,y-s),(x,y+s),(x-s,y-s),(x-s,y),(x-s,y+s)]:
            if (x_i,y_i) in rects:
                edges.append([i,rects[(x_i,y_i)]])
    edges = torch.tensor(edges).T.to(device)
    return features,edges,label


# x,e,_ = get_data("../../BRACS/pkl_data/old/train/BRACS_1003658_2415_1.pth","cpu")
# x.shape,e.shape


In [ ]:
# SlideGraph 节点聚类
from sklearn.neighbors import KDTree as sKDTree
from scipy.cluster import hierarchy
from scipy.cluster.hierarchy import fcluster
from scipy.spatial import Delaunay,KDTree
from collections import defaultdict

def cluster(features,rects):
    lambda_f = 1e-3
    lambda_d = 3e-3
    lambda_h = 0.8
    F = features
    C = rects

    TC = sKDTree(C)
    I,D = TC.query_radius(C,r=6/lambda_d,return_distance=True,sort_results=True)
    DX = np.zeros(int(C.shape[0]*(C.shape[0]-1)/2))
    idx=0
    for i in range(C.shape[0]-1):
        f = np.exp(-lambda_f*np.linalg.norm(F[i]-F[I[i]],axis=1))
        d = np.exp(-lambda_d*D[i])
        df = 1-d*f
        dfi=np.ones(C.shape[0])
        dfi[I[i]] = df
        dfi = dfi[i+1:]
        DX[idx:idx+len(dfi)]=dfi
        idx = idx+len(dfi)
    d = DX
    Z = hierarchy.linkage(d,method="average")
    clusters = fcluster(Z,lambda_h,criterion="distance")
    uc = list(set(clusters))
    C_cluster, F_cluster = [], []
    for c in uc:
        idx = np.where(clusters==c)
        C_cluster.append(list(np.round(C[idx,:].squeeze())) if C[idx,:].squeeze().size==2 else list(np.round(C[idx,:].squeeze().mean(axis=0))))
        F_cluster.append(list(F[idx,:].squeeze()) if C[idx,:].squeeze().size==2 else list(F[idx,:].squeeze().mean(axis=0)))
    C_cluster, F_cluster = np.array(C_cluster), np.array(F_cluster)

    tess = Delaunay(C_cluster)
    neighbors = defaultdict(set)
    for simplex in tess.simplices:
        for idx in simplex:
            other = set(simplex)
            other.remove(idx)
            neighbors[idx] = neighbors[idx].union(other)
    nx = neighbors
    W = np.zeros((C_cluster.shape[0],C_cluster.shape[0]))
    for n in nx:
        nx[n] = np.array(list(nx[n]),dtype=np.int32)
        nx[n] = nx[n][KDTree(C_cluster[nx[n],:]).query_ball_point(C_cluster[n],r=3000)]
        W[n,nx[n]]=1.0
        W[nx[n],n]=1.0
    
    return F_cluster,np.stack(np.where(W>0))

def get_cluster_data(path,device):
    data = torch.load(path)
    features = data["features"].cpu().numpy()
    label = torch.tensor([data["label"]]).to(device)
    rects = np.array(data["rects"])    
    features,edges = cluster(features,rects)

    
    features = torch.tensor(features).to(device)
    edges = torch.tensor(edges).to(device)

    return features,edges,label


class SlideGraph(torch.nn.Module):
    def __init__(self, c_in=384, class_num=2, layers=[16,16,8],pooling='mean',dropout = 0.0,gembed=False,**kwargs):
        super(SlideGraph, self).__init__()
        self.embeddings_dim=layers
        self.no_layers = len(self.embeddings_dim)
        self.first_h = []
        self.nns = []
        self.convs = []
        self.linears = []
        self.pooling = {'max':geom_nn.global_max_pool,'mean':geom_nn.global_mean_pool,'add':geom_nn.global_add_pool}[pooling]
        self.gembed = gembed #if True then learn graph embedding for final classification (classify pooled node features) otherwise pool node decision scores

        for layer, out_emb_dim in enumerate(self.embeddings_dim):
            if layer == 0:
                self.first_h = nn.Sequential(nn.Linear(c_in, out_emb_dim), nn.ReLU())
                self.linears.append(nn.Linear(out_emb_dim, class_num))
                
            else:
                input_emb_dim = self.embeddings_dim[layer-1]
                self.linears.append(nn.Linear(out_emb_dim, class_num))
            
                subnet = nn.Sequential(nn.Linear(2*input_emb_dim, out_emb_dim), nn.ReLU())
                self.nns.append(subnet)                    
                self.convs.append(geom_nn.EdgeConv(self.nns[-1],**kwargs))#DynamicEdgeConv#EdgeConv                aggr='mean'

                    
        self.nns = torch.nn.ModuleList(self.nns)
        self.convs = torch.nn.ModuleList(self.convs)
        self.linears = torch.nn.ModuleList(self.linears)  # has got one more for initial input

    def forward(self, x,edges):
        batch = None
        out = 0
        pooling = self.pooling
        Z = 0
        for layer in range(self.no_layers):            
            if layer == 0:
                x = self.first_h(x)
                z = self.linears[layer](x)
                Z+=z
                dout = pooling(z, batch)
                out += dout
            else:
                x = self.convs[layer-1](x,edges)
                if not self.gembed:
                    z = self.linears[layer](x)
                    Z+=z
                    dout = pooling(z, batch)
                else:
                    dout = self.linears[layer](pooling(x, batch))
                out += dout
        return out,out,Z,out


# import warnings
# warnings.filterwarnings("ignore")
# path = "../../BRACS/pkl_data/old/test/"
# path_save = "../../data/BRACS/test/"
# for f in tqdm(os.listdir(path)):
#     features,edges,label = get_cluster_data(os.path.join(path,f),"cpu")
#     torch.save({"features":features, "label":label, "edges":edges},os.path.join(path_save,f))


# def get_data(path,device):
#     data = torch.load(path)
#     features = data["features"].to(device)
#     label = torch.tensor([data["label"]]).to(device)
#     edges = data["edges"].to(device)
#     return features,edges,label




In [ ]:
# MaxPool
class FCLayer(nn.Module):
    def __init__(self, in_size, out_size=1):
        super(FCLayer, self).__init__()
        self.fc = nn.Sequential(nn.Linear(in_size, out_size))

    def forward(self, feats):
        x = self.fc(feats)
        return feats, x
class MaxPooling(nn.Module):
    def __init__(self,c_in=384,class_num=2):
        super(MaxPooling,self).__init__()
        self.mil=FCLayer(c_in,class_num)

    def forward(self,features,edges):
        results= self.mil(features)[1]
        prediction_i = results
        prediction_b,_ = results.max(0)
        prediction_b = prediction_b.unsqueeze(0)
        attscore = results.argsort(0)
        feature_b = prediction_b
        return prediction_i,prediction_b,attscore,feature_b
# MeanPool
class MeanPooling(nn.Module):
    def __init__(self,c_in=384,class_num=2):
        super(MeanPooling,self).__init__()
        self.mil=FCLayer(c_in,class_num)

    def forward(self,features,edges):
        results= self.mil(features)[1]
        prediction_i = results
        prediction_b = results.mean(0).unsqueeze(0)
        attscore = results.argsort(0)
        feature_b = prediction_b
        return prediction_i,prediction_b,attscore,feature_b

# RNN
class RNN(nn.Module):
    def __init__(self,c_in=384,class_num=2,ndims=128):
        super(RNN, self).__init__()
        self.ndims = ndims

        self.fc1 = nn.Linear(c_in, ndims)
        self.fc2 = nn.Linear(ndims, ndims)

        self.fc3 = nn.Linear(ndims, class_num)

        self.activation = nn.ReLU()

    def init_hidden(self, batch_size):
        return torch.zeros(batch_size, self.ndims)

    def forward(self,features,edges):
        state = self.init_hidden(1).to(features.device)
        for f in features:
            f = f.unsqueeze(0)
            input = self.fc1(f)
            state = self.fc2(state)
            state = self.activation(state+input)
        output = self.fc3(state)
        
        return output,output,output,output

# TransMIL
class TransLayer(nn.Module):
    def __init__(self, norm_layer=nn.LayerNorm, dim=512):
        super().__init__()
        self.norm = norm_layer(dim)
        self.attn = NystromAttention(
            dim = dim,
            dim_head = dim//8,
            heads = 8,
            num_landmarks = dim//2,    # number of landmarks
            pinv_iterations = 6,    # number of moore-penrose iterations for approximating pinverse. 6 was recommended by the paper
            residual = True,         # whether to do an extra residual with the value or not. supposedly faster convergence if turned on
            dropout=0.1
        )
    def forward(self, x):
        x = x + self.attn(self.norm(x))
        return x
class PPEG(nn.Module):
    def __init__(self, dim=512):
        super(PPEG, self).__init__()
        self.proj = nn.Conv2d(dim, dim, 7, 1, 7//2, groups=dim)
        self.proj1 = nn.Conv2d(dim, dim, 5, 1, 5//2, groups=dim)
        self.proj2 = nn.Conv2d(dim, dim, 3, 1, 3//2, groups=dim)

    def forward(self, x, H, W):
        B, _, C = x.shape
        cls_token, feat_token = x[:, 0], x[:, 1:]
        cnn_feat = feat_token.transpose(1, 2).view(B, C, H, W)
        x = self.proj(cnn_feat)+cnn_feat+self.proj1(cnn_feat)+self.proj2(cnn_feat)
        x = x.flatten(2).transpose(1, 2)
        x = torch.cat((cls_token.unsqueeze(1), x), dim=1)
        return x
class TransMIL(nn.Module):
    def __init__(self,c_in=384,class_num=2):
        super(TransMIL, self).__init__()
        self.pos_layer = PPEG(dim=192)
        self._fc1 = nn.Sequential(nn.Linear(c_in, 192), nn.ReLU())
        self.cls_token = nn.Parameter(torch.randn(1, 1, 192))
        self.layer1 = TransLayer(dim=192)
        self.layer2 = TransLayer(dim=192)
        self.norm = nn.LayerNorm(192)
        self._fc2 = nn.Linear(192, class_num)


    def forward(self, features, edges):
        h=features[None,:,:]
        h = self._fc1(h) #[B, n, 512]
        #---->pad
        H = h.shape[1]
        _H, _W = int(np.ceil(np.sqrt(H))), int(np.ceil(np.sqrt(H)))
        add_length = _H * _W - H
        h = torch.cat([h, h[:,:add_length,:]],dim = 1) #[B, N, 512]
        #---->cls_token
        B = h.shape[0]
        cls_tokens = self.cls_token.expand(B, -1, -1).to(features.device)
        h = torch.cat((cls_tokens, h), dim=1)
        #---->Translayer x1
        h = self.layer1(h) #[B, N, 512]
        #---->PPEG
        h = self.pos_layer(h, _H, _W) #[B, N, 512]
        #---->Translayer x2
        h = self.layer2(h) #[B, N, 512]
        #---->cls_token
        h = self.norm(h)[:,0]
        #---->predict
        logits = self._fc2(h) #[B, n_classes]
        
        return logits,logits,logits,logits

# GTP

class GTP(nn.Module):
    def __init__(self, c_in=384, class_num=2):
        super().__init__()

        self.conv1 = geom_nn.GCNConv(c_in, 64)
        self.pool1 = nn.Linear(64, 100)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, 64))

        self.transformers = nn.ModuleList([
            Block(dim=64, num_heads=8, mlp_ratio=2)
            for _ in range(3)
        ])

        self.norm = nn.LayerNorm(64)
        self.head = nn.Linear(64, class_num)

    def forward(self, features, edges):
        x = self.conv1(features, edges)       # [N, 64]
        s = self.pool1(x)                     # [N, 100]

        # 与原dense_mincut_pool默认设置的特征输出一致：
        # 对每个节点的分配分数做softmax，再计算 S^T X。
        # 原训练未使用池化邻接矩阵及两个辅助损失。
        s = F.softmax(s, dim=-1)
        x = (s.transpose(0, 1) @ x).unsqueeze(0)  # [1, 100, 64]

        x = torch.cat([self.cls_token, x], dim=1)

        for block in self.transformers:
            x = block(x)

        x = self.norm(x)[:, 0]
        logits = self.head(x)

        return logits, logits, logits, logits


print("GTP 类已更新：保留原特征池化，移除未使用的稠密计算")
# class GTP(nn.Module):
#     def __init__(self, c_in=384, class_num=2):
#         super(GTP, self).__init__()
#         self.conv1 = geom_nn.GCNConv(c_in,64)       # 384->64
#         self.pool1 = nn.Linear(64, 100)
#         self.cls_token = nn.Parameter(torch.zeros(1, 1, 64))
#         self.transformers = nn.ModuleList([
#             Block(dim=64, num_heads=8, mlp_ratio=2)
#             for i in range(3)])
#         self.norm = nn.LayerNorm(64)
#         self.head = nn.Linear(64, class_num)

#     def forward(self,features, edges):
#         x = self.conv1(features, edges)
#         s = self.pool1(x)
#         adj = torch.zeros((len(x),len(x))).to(x.device)
#         adj[edges[0],edges[1]]=1
#         adj[edges[1],edges[0]]=1
#         adj = adj.unsqueeze(0)
#         x,_,_,_ = geom_nn.dense_mincut_pool(x,adj,s)
#         x = torch.concat([self.cls_token,x],1)
#         for block in self.transformers:
#             x = block(x)
#         x = self.norm(x)[:,0]
#         x = self.head(x)
#         return x,x,x,x

# ABMIL
class ABMIL(nn.Module):
    def __init__(self,c_in=384, class_num=2):
        super().__init__()
        self.att = nn.Sequential(
            nn.Linear(c_in, 64),
            nn.Tanh(),
            nn.Linear(64,1)
        )
        self.classifier = nn.Linear(c_in, class_num)

    def forward(self,features, edges):
        att = F.softmax(self.att(features),dim=0).transpose(1,0)
        x = att@features
        x = self.classifier(x)
        return x,x,x,x

# ===== 新增对比模型：DSMIL =====
class DSMIL_Model(nn.Module):
    def __init__(self, input_size=384, num_classes=2):
        super().__init__()

        # 实例分类分支
        self.instance_fc = nn.Linear(input_size, num_classes)

        # 关系查询映射
        self.q = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.Tanh(),
        )

        # 包级分类分支
        self.fcc = nn.Conv1d(
            num_classes,
            num_classes,
            kernel_size=input_size,
        )

    def forward(self, features):
        # 每个patch的类别分数：N × C
        instance_logits = self.instance_fc(features)

        # 所有patch的查询特征：N × 128
        queries = self.q(features)

        # 每个类别选择一个关键实例
        _, indices = torch.sort(
            instance_logits, dim=0, descending=True
        )
        critical_features = features[indices[0, :]]
        critical_queries = self.q(critical_features)

        # 关系权重：N × C
        attention = queries @ critical_queries.transpose(0, 1)
        attention = F.softmax(
            attention / (queries.shape[1] ** 0.5),
            dim=0,
        )

        # 聚合特征：1 × C × input_size
        bag_features = (
            attention.transpose(0, 1) @ features
        ).unsqueeze(0)

        # 切片分类分数：1 × C
        bag_logits = self.fcc(bag_features).squeeze(-1)

        return instance_logits, bag_logits, attention, bag_features
# ===== 新增对比模型：CLAM-SB =====
class CLAM_SB_Model(nn.Module):
    def __init__(
        self,
        input_size=384,
        num_classes=2,
        dropout=0.25,
        k_sample=8,
    ):
        super().__init__()
        self.num_classes = num_classes
        self.k_sample = k_sample

        # small配置：384 → 512
        self.feature_net = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # 门控注意力的两个分支
        self.attention_a = nn.Sequential(
            nn.Linear(512, 256),
            nn.Tanh(),
            nn.Dropout(dropout),
        )
        self.attention_b = nn.Sequential(
            nn.Linear(512, 256),
            nn.Sigmoid(),
            nn.Dropout(dropout),
        )
        self.attention_c = nn.Linear(256, 1)

        # 单个包表征用于切片分类
        self.bag_classifier = nn.Linear(512, num_classes)

        # 每个类别一个二分类实例约束头
        self.instance_classifiers = nn.ModuleList([
            nn.Linear(512, 2)
            for _ in range(num_classes)
        ])

    def forward(
        self,
        features,
        label=None,
        instance_eval=False,
    ):
        h = self.feature_net(features)       # N × 512

        gated = self.attention_a(h) * self.attention_b(h)
        attention_raw = self.attention_c(gated).transpose(0, 1)
        attention = F.softmax(attention_raw, dim=1)  # 1 × N

        bag_features = attention @ h         # 1 × 512
        bag_logits = self.bag_classifier(bag_features)

        instance_loss = bag_logits.new_zeros(())

        if instance_eval:
            if label is None:
                raise ValueError("实例监督需要切片标签")

            # 避免正负取样重叠，不静默跳过样本
            if h.shape[0] < 2 * self.k_sample:
                raise ValueError(
                    f"CLAM实例取样至少需要"
                    f"{2 * self.k_sample}个patch，"
                    f"当前只有{h.shape[0]}个"
                )

            class_id = int(label.item())
            scores = attention.squeeze(0)

            positive_ids = torch.topk(
                scores, self.k_sample
            ).indices
            negative_ids = torch.topk(
                -scores, self.k_sample
            ).indices

            sampled_features = torch.cat([
                h[positive_ids],
                h[negative_ids],
            ], dim=0)

            # 这是高/低注意力实例的伪标签，
            # 不是人工提供的patch肿瘤标签
            targets = torch.cat([
                torch.ones(
                    self.k_sample,
                    dtype=torch.long,
                    device=h.device,
                ),
                torch.zeros(
                    self.k_sample,
                    dtype=torch.long,
                    device=h.device,
                ),
            ])

            instance_logits = self.instance_classifiers[
                class_id
            ](sampled_features)

            instance_loss = F.cross_entropy(
                instance_logits, targets
            )

        # 统一为四项返回值，便于接入现有循环
        return (
            instance_loss,
            bag_logits,
            attention.transpose(0, 1),
            bag_features,
        )
# model = ABMIL()
# out = model(x,e)
# out.shape
import torch
import torch.nn as nn
import torch_geometric.nn as geom_nn


class SlideGraph(torch.nn.Module):
    def __init__(
        self,
        c_in=384,
        class_num=2,
        layers=[16, 16, 8],
        pooling="mean",
        dropout=0.0,
        gembed=False,
        **kwargs,
    ):
        super(SlideGraph, self).__init__()

        self.embeddings_dim = layers
        self.no_layers = len(self.embeddings_dim)
        self.first_h = []
        self.nns = []
        self.convs = []
        self.linears = []

        self.pooling = {
            "max": geom_nn.global_max_pool,
            "mean": geom_nn.global_mean_pool,
            "add": geom_nn.global_add_pool,
        }[pooling]

        self.gembed = gembed

        for layer, out_emb_dim in enumerate(self.embeddings_dim):
            if layer == 0:
                self.first_h = nn.Sequential(
                    nn.Linear(c_in, out_emb_dim),
                    nn.ReLU(),
                )
                self.linears.append(
                    nn.Linear(out_emb_dim, class_num)
                )
            else:
                input_emb_dim = self.embeddings_dim[layer - 1]

                self.linears.append(
                    nn.Linear(out_emb_dim, class_num)
                )

                subnet = nn.Sequential(
                    nn.Linear(2 * input_emb_dim, out_emb_dim),
                    nn.ReLU(),
                )
                self.nns.append(subnet)

                self.convs.append(
                    geom_nn.EdgeConv(self.nns[-1], **kwargs)
                )

        self.nns = nn.ModuleList(self.nns)
        self.convs = nn.ModuleList(self.convs)
        self.linears = nn.ModuleList(self.linears)

    def forward(self, x, edges):
        batch = None
        out = 0
        Z = 0

        for layer in range(self.no_layers):
            if layer == 0:
                x = self.first_h(x)
                z = self.linears[layer](x)
                Z += z
                out += self.pooling(z, batch)
            else:
                x = self.convs[layer - 1](x, edges)

                if not self.gembed:
                    z = self.linears[layer](x)
                    Z += z
                    dout = self.pooling(z, batch)
                else:
                    dout = self.linears[layer](
                        self.pooling(x, batch)
                    )

                out += dout

        return out, out, Z, out


print("SlideGraph 类已定义")
class RNN(nn.Module):
    def __init__(self, c_in=384, class_num=2, ndims=128):
        super(RNN, self).__init__()
        self.ndims = ndims

        self.fc1 = nn.Linear(c_in, ndims)
        self.fc2 = nn.Linear(ndims, ndims)
        self.fc3 = nn.Linear(ndims, class_num)
        self.activation = nn.ReLU()

    def init_hidden(self, batch_size):
        return torch.zeros(batch_size, self.ndims)

    def forward(self, features, edges):
        state = self.init_hidden(1).to(features.device)

        for f in features:
            f = f.unsqueeze(0)
            input = self.fc1(f)
            state = self.fc2(state)
            state = self.activation(state + input)

        output = self.fc3(state)
        return output, output, output, output


print("原代码 RNN 类已定义")

In [ ]:
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm


# ==================== 配置与划分 ====================

required = ["RNN", "get_data"]
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(f"请先执行这些定义：{missing}")

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

data_dir = Path("/data/database/BRACS/pkl_data/old/train/")
split_source = Path(
    "/data/code/WSI/WSI_pretrain/model/MIL/compare/BRACS/"
    "patience10_split_on_BRACS.json"
)
if not split_source.is_file():
    raise FileNotFoundError(split_source)

split = json.loads(split_source.read_text(encoding="utf-8"))
train_files = split["train"]
val_files = split["val"]

assert len(train_files) == 99
assert len(val_files) == 25
assert len(set(train_files)) == 99
assert len(set(val_files)) == 25
assert set(train_files).isdisjoint(val_files)
assert Path(split["data_path"]).resolve() == data_dir.resolve()

for filename in train_files + val_files:
    if not (data_dir / filename).is_file():
        raise FileNotFoundError(data_dir / filename)

bracs_milrnn_run_dir = Path(
    "/data/code/WSI/WSI_pretrain/model/MIL/temp/"
) / (
    "BRACS_MILRNN_Top10_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)
bracs_milrnn_run_dir.mkdir(parents=True, exist_ok=False)

(
    bracs_milrnn_run_dir / "split_on_BRACS.json"
).write_bytes(split_source.read_bytes())

config = {
    "model": "MILRNN_Top10_DINO",
    "dataset": "BRACS",
    "feature_dim": 384,
    "class_num": 4,
    "hidden_dim": 128,
    "normal_label": 0,
    "top_k": 10,
    "selection_score": "1 - P(class 0)",
    "selection_order": "descending; stable ties",
    "scorer": "Linear(384,4)",
    "scorer_training": (
        "CE on instance with highest predicted non-normal score"
    ),
    "rnn_training": "bag CE; frozen scorer",
    "optimizer": "Adam",
    "learning_rate": 3e-4,
    "max_epochs_per_stage": 100,
    "patience": 10,
    "early_stopping_logic": "reset to 10, then decrement every epoch",
    "consecutive_non_improvements_to_stop": 9,
    "training_seed": None,
    "data_path": str(data_dir),
    "adaptation": "BRACS four-class DINO adaptation",
}

(bracs_milrnn_run_dir / "config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("本次实验目录：", bracs_milrnn_run_dir, flush=True)


# ==================== 评分器与数据读取 ====================

class InstanceScorer(nn.Module):
    def __init__(self, feature_dim=384, class_num=4):
        super().__init__()
        self.fc = nn.Linear(feature_dim, class_num)

    def forward(self, features):
        return self.fc(features)


def load_bracs_slide(filename):
    features, edges, label = get_data(
        str(data_dir / filename), device
    )
    target = label.long().reshape(-1)

    assert features.ndim == 2
    assert features.shape[0] > 0
    assert features.shape[1] == 384
    assert torch.isfinite(features).all()
    assert target.numel() == 1
    assert int(target.item()) in (0, 1, 2, 3)

    return features, edges, target


def abnormal_scores(instance_logits):
    return 1.0 - instance_logits.softmax(dim=1)[:, 0]


def log_message(message):
    print(message, flush=True)
    with (bracs_milrnn_run_dir / "log.log").open(
        "a", encoding="utf-8"
    ) as handle:
        handle.write(message + "\n")


log_message(json.dumps(config, ensure_ascii=False))


# ==================== 公共训练循环 ====================

def train_bracs_stage(stage_name, network, forward_slide):
    optimizer = torch.optim.Adam(
        network.parameters(), lr=3e-4
    )

    best_ce = float("inf")
    best_epoch = None
    remaining = 10
    checkpoint_path = (
        bracs_milrnn_run_dir / f"{stage_name}_best.pth"
    )

    for epoch in range(100):
        network.train()
        train_losses = []

        for filename in tqdm(
            train_files,
            desc=f"{stage_name} Epoch {epoch} Train",
        ):
            features, edges, target = load_bracs_slide(filename)

            optimizer.zero_grad(set_to_none=True)
            logits = forward_slide(features, edges)
            assert tuple(logits.shape) == (1, 4)

            loss = F.cross_entropy(logits, target)
            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"非有限训练损失：{stage_name}, {filename}"
                )

            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        network.eval()
        val_losses = []

        with torch.no_grad():
            for filename in tqdm(
                val_files,
                desc=f"{stage_name} Epoch {epoch} Val",
            ):
                features, edges, target = load_bracs_slide(filename)
                logits = forward_slide(features, edges)
                assert tuple(logits.shape) == (1, 4)

                loss = F.cross_entropy(logits, target)
                if not torch.isfinite(loss):
                    raise RuntimeError(
                        f"非有限验证损失：{stage_name}, {filename}"
                    )

                val_losses.append(loss.item())

        train_ce = float(np.mean(train_losses))
        val_ce = float(np.mean(val_losses))
        improved = val_ce < best_ce

        if improved:
            best_ce = val_ce
            best_epoch = epoch
            remaining = 10

            torch.save(
                {
                    "stage": stage_name,
                    "epoch": epoch,
                    "validation_ce": val_ce,
                    "model_state_dict": network.state_dict(),
                },
                checkpoint_path,
            )

        remaining -= 1

        log_message(
            f"{stage_name} epoch:{epoch} "
            f"remaining:{remaining} "
            f"train:{train_ce:.6f} val:{val_ce:.6f}"
            + (" saved" if improved else "")
        )

        if remaining < 1:
            log_message(f"{stage_name} Early stopping")
            break

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True,
    )
    network.load_state_dict(checkpoint["model_state_dict"])
    network.eval()

    summary = {
        "stage": stage_name,
        "best_epoch": best_epoch,
        "best_validation_ce": best_ce,
        "checkpoint": str(checkpoint_path),
    }
    (
        bracs_milrnn_run_dir / f"{stage_name}_summary.json"
    ).write_text(
        json.dumps(summary, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    log_message(json.dumps(summary, ensure_ascii=False))
    return summary


# ==================== 第一阶段：评分器 ====================

bracs_scorer = InstanceScorer(384, 4).to(device)


def scorer_forward_slide(features, edges):
    logits = bracs_scorer(features)

    # 不根据真实类别选实例
    index = abnormal_scores(logits).argmax()
    return logits[index].unsqueeze(0)


scorer_summary = train_bracs_stage(
    "scorer",
    bracs_scorer,
    scorer_forward_slide,
)

bracs_scorer.eval()
for parameter in bracs_scorer.parameters():
    parameter.requires_grad_(False)


# ==================== 第二阶段：原RNN ====================

bracs_rnn = RNN(
    c_in=384,
    class_num=4,
    ndims=128,
).to(device)


def rnn_forward_slide(features, edges):
    with torch.no_grad():
        scores = abnormal_scores(bracs_scorer(features))
        indices = torch.argsort(
            scores, descending=True, stable=True
        )[:min(10, features.shape[0])]
        selected_features = features[indices]

    # 原RNN不使用空间边
    _, bag_logits, _, _ = bracs_rnn(selected_features, None)
    return bag_logits


rnn_summary = train_bracs_stage(
    "rnn",
    bracs_rnn,
    rnn_forward_slide,
)


# ==================== 保存实例选择记录 ====================

selection_records = {}

with torch.no_grad():
    for filename in tqdm(
        train_files + val_files,
        desc="保存BRACS Top-10记录",
    ):
        features, _, _ = load_bracs_slide(filename)
        scores = abnormal_scores(bracs_scorer(features))
        indices = torch.argsort(
            scores, descending=True, stable=True
        )[:min(10, features.shape[0])]

        selection_records[filename] = {
            "indices": indices.cpu().tolist(),
            "abnormal_scores": scores[indices].cpu().tolist(),
        }

(bracs_milrnn_run_dir / "selected_instances.json").write_text(
    json.dumps(selection_records, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

(bracs_milrnn_run_dir / "training_summary.json").write_text(
    json.dumps(
        {
            "model": "BRACS_MILRNN_Top10_DINO",
            "scorer": scorer_summary,
            "rnn": rnn_summary,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\n训练完成，保存目录：", bracs_milrnn_run_dir)

In [ ]:
import json
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import (
    precision_score,
    accuracy_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

# 重启内核后填写本次目录，并重新定义InstanceScorer、RNN、get_data
experiment_folder = ""

if experiment_folder:
    run_dir = Path(experiment_folder)
elif "bracs_milrnn_run_dir" in globals():
    run_dir = Path(bracs_milrnn_run_dir)
else:
    raise ValueError("请填写 experiment_folder")

for filename in (
    "config.json",
    "training_summary.json",
    "scorer_best.pth",
    "rnn_best.pth",
):
    if not (run_dir / filename).is_file():
        raise FileNotFoundError(run_dir / filename)

config = json.loads(
    (run_dir / "config.json").read_text(encoding="utf-8")
)
assert config["dataset"] == "BRACS"
assert config["class_num"] == 4
assert config["selection_score"] == "1 - P(class 0)"

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

test_scorer = InstanceScorer(
    config["feature_dim"], config["class_num"]
).to(device)

test_rnn = RNN(
    c_in=config["feature_dim"],
    class_num=config["class_num"],
    ndims=config["hidden_dim"],
).to(device)

scorer_checkpoint = torch.load(
    run_dir / "scorer_best.pth",
    map_location=device,
    weights_only=True,
)
rnn_checkpoint = torch.load(
    run_dir / "rnn_best.pth",
    map_location=device,
    weights_only=True,
)

test_scorer.load_state_dict(
    scorer_checkpoint["model_state_dict"]
)
test_rnn.load_state_dict(
    rnn_checkpoint["model_state_dict"]
)
test_scorer.eval()
test_rnn.eval()

print("本次实验目录：", run_dir.resolve())
print("评分器最佳轮次：", scorer_checkpoint["epoch"])
print("RNN最佳轮次：", rnn_checkpoint["epoch"])

test_dir = Path("/data/database/BRACS/pkl_data/old/test/")
test_files = sorted(
    p for p in test_dir.iterdir()
    if p.is_file() and p.suffix == ".pth"
)
assert len(test_files) == 31

labels, predictions, rows = [], [], []

with torch.no_grad():
    for path in tqdm(test_files, desc="BRACS MIL-RNN Top10 Test"):
        features, edges, label = get_data(str(path), device)

        assert features.ndim == 2
        assert features.shape[0] > 0
        assert features.shape[1] == config["feature_dim"]
        assert torch.isfinite(features).all()

        instance_logits = test_scorer(features)
        assert torch.isfinite(instance_logits).all()

        scores = 1.0 - instance_logits.softmax(dim=1)[:, 0]
        indices = torch.argsort(
            scores, descending=True, stable=True
        )[:min(config["top_k"], features.shape[0])]

        _, bag_logits, _, _ = test_rnn(features[indices], None)

        assert tuple(bag_logits.shape) == (1, 4)
        assert torch.isfinite(bag_logits).all()

        true_label = int(label.item())
        assert true_label in (0, 1, 2, 3)
        prediction = int(bag_logits.argmax(dim=1).item())

        labels.append(true_label)
        predictions.append(prediction)
        rows.append({
            "file": path.name,
            "label": true_label,
            "prediction": prediction,
            "class_probabilities": (
                bag_logits.softmax(dim=1)[0].cpu().tolist()
            ),
            "selected_indices": indices.cpu().tolist(),
            "selected_abnormal_scores": (
                scores[indices].cpu().tolist()
            ),
        })

distribution = [labels.count(i) for i in range(4)]
assert distribution == [12, 6, 7, 6], distribution

class_labels = [0, 1, 2, 3]

result = {
    "model": "BRACS_MILRNN_Top10_DINO",
    "run_id": run_dir.name,
    "scorer_weight_path": str(
        (run_dir / "scorer_best.pth").resolve()
    ),
    "rnn_weight_path": str(
        (run_dir / "rnn_best.pth").resolve()
    ),
    "n": len(labels),
    "average": "macro",
    "top_k": config["top_k"],
    "selection_score": config["selection_score"],
    "precision": float(precision_score(
        labels, predictions,
        labels=class_labels,
        average="macro",
        zero_division=0,
    )),
    "accuracy": float(accuracy_score(labels, predictions)),
    "recall": float(recall_score(
        labels, predictions,
        labels=class_labels,
        average="macro",
        zero_division=0,
    )),
    "f1": float(f1_score(
        labels, predictions,
        labels=class_labels,
        average="macro",
        zero_division=0,
    )),
    "confusion_matrix_labels_0_1_2_3": confusion_matrix(
        labels, predictions, labels=class_labels
    ).tolist(),
}

summary = (
    f"TEST {result['model']} | "
    f"Pre: {result['precision']:.3f}   "
    f"Acc: {result['accuracy']:.3f}   "
    f"Rec: {result['recall']:.3f}   "
    f"F1: {result['f1']:.3f}"
)

print(summary)
print("混淆矩阵：行是真实类别，列是预测类别；顺序0、1、2、3")
print(np.array(result["confusion_matrix_labels_0_1_2_3"]))

for filename, content in (
    ("test_metrics.json", result),
    ("test_predictions.json", rows),
):
    with (run_dir / filename).open(
        "w", encoding="utf-8"
    ) as handle:
        json.dump(content, handle, ensure_ascii=False, indent=2)

log_path = run_dir / "log.log"
with log_path.open("a", encoding="utf-8") as handle:
    handle.write("\n" + summary + "\n")
    handle.write(
        str(result["confusion_matrix_labels_0_1_2_3"]) + "\n"
    )

assert summary in log_path.read_text(encoding="utf-8")
print("测试结果和日志已保存至：", run_dir.resolve())

In [ ]:
import numpy as np
import torch
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score

required = ["model", "split", "train_dir", "device", "get_data"]
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(f"请先运行之前的权重与划分加载代码：{missing}")

model.eval()

labels = []
predictions = {
    "原序": [],
    "逆序": [],
    "固定随机顺序": [],
}
probabilities = {name: [] for name in predictions}

# 局部随机数生成器，不改变训练用的全局随机状态
generator = torch.Generator(device="cpu")
generator.manual_seed(42)

with torch.no_grad():
    for filename in tqdm(split["val"], desc="验证集顺序诊断"):
        features, edges, label = get_data(
            str(train_dir / filename), device
        )
        labels.append(int(label.item()))

        permutation = torch.randperm(
            features.size(0), generator=generator
        ).to(features.device)

        variants = {
            "原序": features,
            "逆序": features.flip(0),
            "固定随机顺序": features[permutation],
        }

        for name, inputs in variants.items():
            # 原RNN不使用edges，本诊断仅适用于该RNN
            _, logits, _, _ = model(inputs, edges)
            assert tuple(logits.shape) == (1, 2)
            assert torch.isfinite(logits).all()

            predictions[name].append(
                int(logits.argmax(dim=1).item())
            )
            probabilities[name].append(
                float(logits.softmax(dim=1)[0, 1].item())
            )

original_predictions = np.array(predictions["原序"])
original_probabilities = np.array(probabilities["原序"])

for name in predictions:
    pred = np.array(predictions[name])
    prob = np.array(probabilities[name])

    print(f"\n{name}")
    print(f"Accuracy：{accuracy_score(labels, pred):.4f}")
    print(f"F1：{f1_score(labels, pred, zero_division=0):.4f}")
    print(
        "相对原序的预测改变比例："
        f"{np.mean(pred != original_predictions):.4f}"
    )
    print(
        "相对原序的平均肿瘤概率绝对变化："
        f"{np.mean(np.abs(prob - original_probabilities)):.4f}"
    )

In [ ]:
device = torch.device("cuda:0")
# class_num = 4
class_num = 2

# model = MeanPooling(384,class_num).to(device)
# model = MaxPooling(384,class_num).to(device)
# model = RNN(384,class_num).to(device)
# model = TransMIL(384,class_num).to(device)
# model = GTP(384,class_num).to(device)
# model = ABMIL(384,class_num).to(device)
model = DSMIL_Model(384,class_num,).to(device)
# model = CLAM_SB_Model(384,class_num,
#     dropout=0.25,
#     k_sample=8,
# ).to(device)
loss_func = nn.CrossEntropyLoss()
optim = torch.optim.Adam(model.parameters(),3e-4)

#云GPU训练路径
treain_data_path = ( "/data/database/CAMELYON16/pkl_data/training/20xbags/")
# treain_data_path = "../../CAMELYON16/pkl_data/training/20xbags/"
# treain_data_path = "/data/database/BRACS/pkl_data/old/train/"
# os.makedirs(save_path, exist_ok=True)
train_data_files = np.array(os.listdir(treain_data_path))
train_data_files,val_data_files = train_test_split(train_data_files,test_size=0.2,random_state=42)
#云GPU训练路径
# save_path = "./model/MIL/compare/Camelyon/DSMIL/"
# os.makedirs(save_path, exist_ok=False)
# save_path = "./model/MIL/compare/Camelyon/CLAM_SB/"
save_path = "./model/MIL/compare/temp"
# save_path = "./model/MIL/compare/BRACS/CLAM_SB/"
os.makedirs(save_path, exist_ok=True)

# # 防止重复运行覆盖已有结果
# for filename in ("model.pth", "log.log"):
#     path = os.path.join(save_path, filename)
#     if os.path.exists(path):
#         raise FileExistsError(f"已有结果，请先检查：{path}")

logger = logging.getLogger('logger')
logger.setLevel(logging.INFO)
logger.handlers=[]
fh = logging.FileHandler(os.path.join(save_path,'log.log'),"w")
fh.setLevel(logging.INFO)
formatter = logging.Formatter('%(asctime)s - %(message)s')
fh.setFormatter(formatter)
logger.addHandler(fh)


loss_min = np.inf
# patience = 5
patience = 10
e_count = patience
for e in range(100):
    model.train()
    train_loss = []
    for f in tqdm(train_data_files):
        features,edges,label = get_data(os.path.join(treain_data_path,f),device)
        # if len(edges)<1:
        #     continue
        # prediction_i, prediction_b, attscore, feature_b = model(features,edges)
        
        # optim.zero_grad()
        # loss = loss_func(prediction_b,label)
        # loss.backward()
        # optim.step()
        # # DSMIL只接收patch特征，不使用空间边
        prediction_i, prediction_b, attscore, feature_b = model(features)
        
        # 切片标签转换成两个类别的目标向量
        target = F.one_hot(label.long(), num_classes=class_num).float()
        
        # 每个类别在所有patch中的最高实例分数
        max_instance_logits = prediction_i.max(
            dim=0, keepdim=True
        ).values
        
        # DSMIL联合损失：包级分支＋最大实例分支
        loss = (
            0.5 * F.binary_cross_entropy_with_logits(prediction_b, target)
            + 0.5 * F.binary_cross_entropy_with_logits(
                max_instance_logits, target
            )
        )
        # # 训练时启用实例约束
        # instance_loss, prediction_b, attscore, feature_b = model(
        #     features,
        #     label=label.long(),
        #     instance_eval=True,
        # )
        
        # bag_loss = F.cross_entropy(prediction_b, label.long())
        
        # # CLAM-SB联合损失
        # loss = 0.7 * bag_loss + 0.3 * instance_loss
        optim.zero_grad()
        loss.backward()
        optim.step()
        train_loss.append(loss.item())

    model.eval()
    val_loss = []

    for f in tqdm(val_data_files):
        with torch.no_grad():
            features, edges, label = get_data(
                os.path.join(treain_data_path, f), device
            )

            # _, prediction_b, attscore, feature_b = model(
            #     features,
            #     instance_eval=False,
            # )
            _, prediction_b, attscore, feature_b = model(features)
            loss = F.cross_entropy(
                prediction_b, label.long()
            )
            val_loss.append(loss.item())

    # 所有验证样本完成后，再计算均值和保存权重
    train_loss = np.mean(train_loss)
    val_loss = np.mean(val_loss)

    info = (
        f"{e}/{e_count}\ttrain:{train_loss}"
        f"\tval:{val_loss}\t"
    )

    if val_loss < loss_min:
        e_count = patience
        loss_min = val_loss
        info += "saved"
        torch.save(
            model, os.path.join(save_path, "model.pth")
        )

    e_count -= 1

    logger.info(info)
    print(info)

    if e_count < 1:
        logger.info("Early stopping")
        print("Early stopping")
        break



In [ ]:
from pathlib import Path

script = Path("retrain_baselines_patience10.py")

print("脚本路径：", script.resolve(), flush=True)
print("脚本存在：", script.is_file(), flush=True)

required = ["TransMIL", "get_data"]
missing = [
    name for name in required
    if name not in globals()
]

if missing:
    raise NameError(
        f"请先执行这些类或函数的定义：{missing}"
    )

print("准备启动TransMIL训练", flush=True)

%run -i retrain_baselines_patience10.py --models TransMIL

print("TransMIL训练结束", flush=True)

In [ ]:
import os
import json
import numpy as np
import torch
from tqdm import tqdm
from sklearn.metrics import (
    precision_score,
    accuracy_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

save_path = (
    "./model/MIL/compare/Camelyon/"
    "TransMIL/TransMIL_patience10/"
)
test_data_path = (
    "/data/database/CAMELYON16/pkl_data/testing/20xbags/"
)

test_files = sorted(
    filename
    for filename in os.listdir(test_data_path)
    if filename.endswith(".pth")
)

assert len(test_files) == 129, (
    f"预期129张测试切片，实际{len(test_files)}张"
)

# 先执行TransMIL及其依赖类、get_data的定义
model = torch.load(
    os.path.join(save_path, "model.pth"),
    map_location=device,
    weights_only=False,
).to(device)
model.eval()

labels = []
predictions = []
rows = []

with torch.no_grad():
    for filename in tqdm(
        test_files, desc="TransMIL Test"
    ):
        features, edges, label = get_data(
            os.path.join(test_data_path, filename),
            device,
        )

        # 原始TransMIL接口接收edges，但内部不使用空间边
        _, bag_logits, _, _ = model(features, edges)

        assert tuple(bag_logits.shape) == (1, 2)
        assert torch.isfinite(bag_logits).all()

        true_label = int(label.item())
        prediction = int(
            bag_logits.argmax(dim=1).item()
        )

        labels.append(true_label)
        predictions.append(prediction)
        rows.append({
            "file": filename,
            "label": true_label,
            "prediction": prediction,
        })

assert [labels.count(0), labels.count(1)] == [81, 48], (
    "测试集标签分布与此前统计不一致"
)

result = {
    "model": "CAMELYON16_TransMIL_patience10",
    "n": len(labels),
    "positive_label": 1,
    "precision": float(precision_score(
        labels, predictions,
        pos_label=1, zero_division=0,
    )),
    "accuracy": float(accuracy_score(
        labels, predictions,
    )),
    "recall": float(recall_score(
        labels, predictions,
        pos_label=1, zero_division=0,
    )),
    "f1": float(f1_score(
        labels, predictions,
        pos_label=1, zero_division=0,
    )),
    "confusion_matrix": confusion_matrix(
        labels, predictions, labels=[0, 1]
    ).tolist(),
}

summary = (
    f"TEST {result['model']} | "
    f"Pre: {result['precision']:.3f}   "
    f"Acc: {result['accuracy']:.3f}   "
    f"Rec: {result['recall']:.3f}   "
    f"F1: {result['f1']:.3f}"
)

print(summary)
print("混淆矩阵：行是真实类别，列是预测类别")
print(np.array(result["confusion_matrix"]))

with open(
    os.path.join(save_path, "test_metrics.json"),
    "w", encoding="utf-8",
) as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

with open(
    os.path.join(save_path, "test_predictions.json"),
    "w", encoding="utf-8",
) as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)

# 追加测试结果，不覆盖训练日志
with open(
    os.path.join(save_path, "log.log"),
    "a", encoding="utf-8",
) as f:
    f.write("\n" + summary + "\n")
    f.write("Confusion matrix [0, 1]:\n")
    for row in result["confusion_matrix"]:
        f.write(str(row) + "\n")

In [ ]:
device = torch.device("cuda:0")

# test_data_path = "/data/database/CAMELYON16/pkl_data/testing/20xbags/"
test_data_path = "/data/database/BRACS/pkl_data/old/test/"
test_data_files = np.array(os.listdir(test_data_path))

# model = torch.load(
#     "./model/MIL/Ablation/MIL_20/model.pth",
#     map_location=device,
#     weights_only=False,
# )
# model = torch.load(
#     "./model/MIL/compare/BRACS/DSMIL/model.pth",
#     map_location=device,
#     weights_only=False,
# )
model = torch.load(
    "./model/MIL/compare/BRACS/CLAM_SB/model.pth",
    map_location=device,
    weights_only=False,
)
# model = torch.load("./model/MIL/Ablation/MIL_20/model.pth",weights_only=False)
# model = torch.load("./model/MIL/compare/BRACS/temp/model.pth",weights_only=False)
model.eval()
pred_list = []
label_list = []
model.eval()
for f in tqdm(test_data_files[:]):
    with torch.no_grad():
        features,edges,label = get_data(os.path.join(test_data_path,f),device)

        # prediction_i, prediction_b, attscore, feature_b = model(features,edges)
        prediction_i, prediction_b, attscore, feature_b = model(features)
        prediction_b = torch.sigmoid(prediction_b).cpu().numpy()

        label_list.append(label.item())
        pred_list.append(prediction_b)
label = np.array(label_list)
pred = np.array(pred_list)
# pred = pred.argmax(-1
pred = pred.argmax(-1).reshape(-1)
# print(f"Pre: {precision_score(label,pred):.3f}   Acc: {accuracy_score(label,pred):.3f}   Rec: {recall_score(label,pred):.3f}   F1: {f1_score(label,pred):.3f}")
print(f"Pre: {precision_score(label,pred,average='macro'):.3f}   Acc: {accuracy_score(label,pred):.3f}   Rec: {recall_score(label,pred,average='macro'):.3f}   F1: {f1_score(label,pred,average='macro'):.3f}")

In [ ]:
# pre = 0.889
# rec = 0.667
# 2*pre*rec/(pre+rec)
with open(
    "./model/MIL/compare/BRACS/CLAM_SB/log.log",
    "a",
    encoding="utf-8",
) as f:
    f.write(
        "\nTEST CLAM_SB | "
        "Pre: 0.812   Acc: 0.839   Rec: 0.804   F1: 0.792\n"
    )

In [ ]:
device = torch.device("cpu")

weight_path = (
    "./model/MIL/compare/BRACS/MeanPool/model.pth"
)

# 只加载可信的原作者权重
model = torch.load(
    weight_path,
    map_location=device,
    weights_only=False,
)
model.eval()

folders = {
    "旧版 old/test":
        "/data/database/BRACS/pkl_data/old/test/",
    "另一版 testing":
        "/data/database/BRACS/pkl_data/testing/",
}

for name, folder in folders.items():
    print(f"\n===== {name} =====")

    if not os.path.isdir(folder):
        print("目录不存在：", folder)
        continue

    files = sorted(
        f for f in os.listdir(folder)
        if f.endswith(".pth")
    )

    if not files:
        print("未找到pth文件")
        continue

    labels = []
    predictions = []

    with torch.no_grad():
        for filename in tqdm(files):
            data = torch.load(
                os.path.join(folder, filename),
                map_location="cpu",
                weights_only=False,
            )

            features = data["features"].float()
            label = int(data["label"])

            assert features.ndim == 2
            assert features.shape[1] == 384
            assert features.shape[0] > 0
            assert torch.isfinite(features).all()
            assert label in (0, 1, 2, 3)

            # 原MeanPool接口保留edges参数，
            # 但内部不使用，因此传None即可
            _, logits, _, _ = model(features, None)

            assert tuple(logits.shape) == (1, 4)
            assert torch.isfinite(logits).all()

            labels.append(label)
            predictions.append(int(logits.argmax(1).item()))

    print("样本数：", len(labels))
    print("标签分布：", dict(sorted(Counter(labels).items())))

    print(
        f"Pre: {precision_score(labels, predictions, average='macro', zero_division=0):.3f}   "
        f"Acc: {accuracy_score(labels, predictions):.3f}   "
        f"Rec: {recall_score(labels, predictions, average='macro', zero_division=0):.3f}   "
        f"F1: {f1_score(labels, predictions, average='macro', zero_division=0):.3f}"
    )

    print("混淆矩阵，行是真实类别，列是预测类别：")
    print(confusion_matrix(
        labels, predictions, labels=[0, 1, 2, 3]
    ))

In [ ]:
from pathlib import Path
from collections import Counter
import torch

base = Path("/data/database/BRACS/pkl_data/old")
counts = {}

for part in ("train", "test"):
    files = sorted((base / part).glob("*.pth"))
    counter = Counter()

    for path in files:
        data = torch.load(
            path, map_location="cpu", weights_only=False
        )
        counter[int(data["label"])] += 1

    counts[part] = counter
    print(
        part,
        "文件数：", len(files),
        "标签分布：", dict(sorted(counter.items()))
    )

print("\n集合\t标签0\t标签1\t标签2\t标签3\t总计")
for part in ("train", "test"):
    c = counts[part]
    print(
        part,
        *(c[i] for i in range(4)),
        sum(c.values()),
        sep="\t",
    )

total = counts["train"] + counts["test"]
print(
    "总计",
    *(total[i] for i in range(4)),
    sum(total.values()),
    sep="\t",
)

In [ ]:
from pathlib import Path

base = Path("/data/database/BRACS")

for path in base.rglob("*"):
    if path.is_file() and path.suffix.lower() in {
        ".csv", ".xlsx", ".xls", ".tsv", ".json", ".xml"
    }:
        print(path)

In [ ]:
# # Camelyon
# device = torch.device("cuda:0")
# model = torch.load("./model/MIL/Ablation/GCN_3/model.pth")
# path = "../../CAMELYON16/pkl_data/testing/20xbags/test_040_3239_1.pth"

# features,edges,label = get_data(path,device)
# data = torch.load(path)
# rects = data["rects"]
# level = data["level"]
# level_base = data["level_base"]
# p_size = data["patch_size"]

# wsi = WSI("../../CAMELYON16/testing/images/test_040.tif")
# mask,_ = wsi.get_mask("../../CAMELYON16/label/testing/test_040.xml",level_base)
    


# prediction_i, prediction_b, attscore, feature_b = model(features,edges)
# prediction_b = torch.sigmoid(prediction_b)[:, -1]



# p = 2**(level_base-level)
# w_size = p_size//p
# img = wsi.get_image(level_base)
# mask_pred = np.zeros((img.shape[0],img.shape[1]))

# # attscore = attscore[0]
# att = (attscore-attscore.min())/(attscore.max()-attscore.min())

# for (x,y),v in zip(rects,att):
#     x,y = x//p,y//p
#     cv.rectangle(img,(y,x),(y+w_size,x+w_size),(0,0,0))
#     mask_pred[x:x+w_size,y:y+w_size] = v[1].item()
#     # mask_pred[x:x+w_size,y:y+w_size] = v.item()

#     # break

# print(label,prediction_b)

# plt.figure(figsize=(18,12))
# plt.subplot(1,3,1)
# plt.axis("off")
# plt.imshow(img)

# plt.subplot(1,3,2)
# plt.axis("off")
# plt.imshow(mask)

# mask_pred[mask_pred>0.5]=1
# mask_pred[mask_pred<1]=0
# plt.subplot(1,3,3)
# plt.axis("off")
# plt.imshow(mask_pred,cmap="coolwarm")
# plt.show()

# rec,iou = ((mask_pred*mask)>0).sum()/((mask)>0).sum(), ((mask_pred*mask)>0).sum()/((mask_pred+mask)>0).sum()
# iou,rec

In [ ]:
# # BACH BRACS
# device = torch.device("cuda:0")
# model = torch.load("./model/MIL/temp/model.pth")
# # path = "../../BACH/pkl_data/training/20xbags/A01_896_0.pth"
# path = "../../BRACS/pkl_data/test/BRACS_292_1534_3.pth"

# features,edges,label = get_data(path,device)
# data = torch.load(path)
# rects = data["rects"]
# level = data["level"]
# level_base = data["level_base"]
# p_size = data["patch_size"]

# # wsi = WSI("../../BACH/WSI/train/wsi/A01.svs")
# wsi = WSI("D:/TJL/Invasive/BRACS_292.svs")


# prediction_i, prediction_b, attscore, feature_b = model(features,edges)
# prediction_b = torch.sigmoid(prediction_b)



# p = 4**(level_base-level)
# w_size = p_size//p
# img = wsi.get_image(level_base)
# mask_pred = np.zeros((img.shape[0],img.shape[1]))

# # attscore = attscore[0]
# att = (attscore-attscore.min())/(attscore.max()-attscore.min())

# for (x,y),v in zip(rects,att):
#     x,y = x//p,y//p
#     cv.rectangle(img,(y,x),(y+w_size,x+w_size),(0,0,0))
#     mask_pred[x:x+w_size,y:y+w_size] = v[2].item()

# print(label,prediction_b,prediction_b.argmax())

# plt.figure(figsize=(18,12))
# plt.subplot(1,2,1)
# plt.axis("off")
# plt.imshow(img)

# plt.subplot(1,2,2)
# plt.axis("off")
# plt.imshow(mask_pred,cmap="coolwarm")
# plt.show()
